In [2]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

PDF를 처리하기 위해 먼저 `pypdf` 라이브러리를 설치합니다. 이 라이브러리를 통해 PDF 문서에서 텍스트를 추출할 수 있습니다.

In [3]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 6.7 MB/s eta 0:00:00


이제 PDF 파일을 열고 텍스트 내용을 추출하겠습니다.

In [4]:
from pypdf import PdfReader

file_path = '/content/모델배포기초6.pdf'

reader = PdfReader(file_path)
num_pages = len(reader.pages)

full_text = []
for i in range(num_pages):
    page = reader.pages[i]
    full_text.append(page.extract_text())

# Join all page texts into a single string
extracted_text = '\n'.join(full_text)

print(f'Successfully extracted text from {num_pages} pages.')
print('\n--- First 500 characters of the extracted text ---\n')
print(extracted_text[:500])
print('\n----------------------------------------------------\n')

Successfully extracted text from 33 pages.

--- First 500 characters of the extracted text ---

DAY 06 · 모델  배포  개론
인증과
미디어  처리  기초
누구나  부르던  API 에  자물쇠를  채우고 , 들어오는  파일은  문  앞에서  검사한다
오늘의  한  문장  — 열린  API 는  공짜  GPU입니다 . 누구인지  확인하고 ( 인증 ), 무엇을
허락할지  정하고 ( 인가 ), 오가는  길은  암호화하고 (TLS), 비밀은  코드  밖에  두고 ( 시크릿 ),
모든  출입은  기록합니다 ( 로그 ). 그  위에서  이미지를  안전하게 받아  추론까지  잇습니다 .
과정 모델  배포  개론 작성 김지성  · 모두의연구소 범위 §1 ~ §6
01 / 33
Day 6 Learning Goals
오늘  끝내면  할  수  있는  것
네  가지면  충분합니다 .
01
왜  자물쇠인가
인증  없는  추론  API 에  벌어지는  일  — 서버  다운 · 비용  폭탄 · 모델  도용 · 추적  불가  — 을
시나리오로  설명하고 , API Key를  고르는  이유를  말할 

----------------------------------------------------



추출된 텍스트를 텍스트 파일(.txt)로 저장합니다.

In [5]:
output_file_name = 'extracted_pdf_content.txt'

with open(output_file_name, 'w', encoding='utf-8') as f:
    f.write(extracted_text)

print(f'Successfully saved the extracted text to {output_file_name}')

Successfully saved the extracted text to extracted_pdf_content.txt


### 구현: 안전한 모델 배포 API

'Day 6' 학습 목표에 따라 다음 기능을 구현합니다:
1. **인증(Authentication)**: 정적 API Key를 사용하여 API를 '잠금' 처리합니다.
2. **미디어 처리**: 추론을 위해 파일(이미지)을 안전하게 수신합니다.
3. **로깅**: 접속 기록을 추적하기 위한 기본 로그를 생성합니다.

In [6]:
from fastapi import FastAPI, Depends, HTTPException, Security, File, UploadFile
from fastapi.security.api_key import APIKeyHeader
import starlette.status as status

app = FastAPI(title="Secure Inference API")

# 1. Security Configuration (Authentication)
API_KEY = "my_super_secret_key"
API_KEY_NAME = "access_token"
api_key_header = APIKeyHeader(name=API_KEY_NAME, auto_error=False)

async def get_api_key(header_value: str = Security(api_key_header)):
    if header_value == API_KEY:
        return header_value
    raise HTTPException(
        status_code=status.HTTP_403_FORBIDDEN, detail="Could not validate credentials"
    )

# 2. Secure Inference Endpoint with Media Handling
@app.post("/predict")
async def predict(file: UploadFile = File(...), api_key: str = Depends(get_api_key)):
    # Log the access (Audit Log concept)
    print(f"Access granted for file: {file.filename}")

    # Simulate model inference
    content = await file.read()
    return {
        "filename": file.filename,
        "content_type": file.content_type,
        "status": "Inference Successful",
        "result": "Example Model Output"
    }

print("FastAPI structure for secure deployment is ready.")

FastAPI structure for secure deployment is ready.


### [장별 구현 1] 인증(Authentication) 및 인가(Authorization)
문서 초반부(§1~§2)에서 강조하는 '자물쇠(API Key)'와 '권한 관리'를 구현합니다.

In [7]:
import jwt
from datetime import datetime, timedelta

SECRET_KEY = "super_secret_deployment_key"
ALGORITHM = "HS256"

# JWT 생성 함수 (로그인 및 토큰 발급 단계)
def create_access_token(data: dict):
    to_encode = data.copy()
    expire = datetime.utcnow() + timedelta(minutes=15)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

# 사용자 역할(Role) 정의 (인가 단계)
USER_DB = {"admin_user": {"role": "admin"}, "normal_user": {"role": "guest"}}

def check_admin_permission(token):
    payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    username = payload.get("sub")
    if USER_DB.get(username, {}).get("role") != "admin":
        raise PermissionError("관리자 권한이 필요합니다.")
    return True

### [장별 구현 2] 보안 미디어 처리 (Secure Media Handling)
문서 중반부(§3~§4)의 '이미지를 안전하게 받아 추론'하는 과정을 검증 로직과 함께 구현합니다.

In [10]:
!apt-get install -y libmagic-dev
!pip install python-magic

import magic

def validate_image_file(file_bytes):
    # 문서에서 강조한 '문 앞 검사': 파일 확장자만이 아닌 실제 매직 넘버 확인
    mime = magic.Magic(mime=True)
    file_type = mime.from_buffer(file_bytes)

    ALLOWED_TYPES = ["image/jpeg", "image/png"]
    if file_type not in ALLOWED_TYPES:
        return False, f"허용되지 않는 파일 형식: {file_type}"

    # 파일 크기 제한 (비용 폭탄 및 DoS 방어)
    if len(file_bytes) > 5 * 1024 * 1024: # 5MB 제한
        return False, "파일 크기가 너무 큽니다."

    return True, "검증 완료"

# 테스트 실행 (PNG 매직 넘버 시뮬레이션 데이터)
test_data = b"\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR"
is_valid, msg = validate_image_file(test_data)
print(f"결과: {is_valid}, 메시지: {msg}")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  libmagic-dev
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 105 kB of archives.
After this operation, 389 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libmagic-dev amd64 1:5.41-3ubuntu0.1 [105 kB]
Fetched 105 kB in 0s (573 kB/s)
Selecting previously unselected package libmagic-dev:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libmagic-dev_1%3a5.41-3ubuntu0.1_amd64.deb ...
Unpacking libmagic-dev:amd64 (1:5.41-3ubuntu0.1) ...
Setting up libmagic-dev:amd64 (1:5.41-3ubuntu0.1) ...
Processing triggers for man-db (2.10.2-1) ...
결과: True, 메시지: 검증 완료


### [장별 구현 3] 로깅 및 모니터링 (Logging & Audit)
문서 후반부(§5~§6)의 '모든 출입 기록'을 위한 로깅 시스템을 구현합니다.

In [9]:
import logging

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("DeploymentAudit")

def audit_log(user_id, action, status):
    timestamp = datetime.now().isoformat()
    log_msg = f"[{timestamp}] User: {user_id} | Action: {action} | Status: {status}"
    logger.info(log_msg)
    # 실제 운영 환경에서는 이를 DB나 ELK 스택으로 전송

audit_log("user_123", "inference_request", "success")

### [심화] 운영 환경을 위한 비동기 로깅 (Asynchronous Logging)
메인 로직(추론)의 속도를 방해하지 않도록 로그 생성을 별도 쓰레드에서 처리하는 비동기 구조를 구현합니다.

In [11]:
import logging
import logging.handlers
import queue
import threading
import time

# 1. 로그를 담을 큐 생성
log_queue = queue.Queue(-1) # 무제한 크기

# 2. 실제로 로그를 기록할 핸들러 설정 (파일 혹은 콘솔)
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(message)s')
handler.setFormatter(formatter)

# 3. 큐에서 로그를 읽어 실제 핸들러로 전달하는 리스너 설정
listener = logging.handlers.QueueListener(log_queue, handler)
listener.start()

# 4. 앱에서 사용할 로거 설정 (QueueHandler 연결)
async_logger = logging.getLogger("AsyncAudit")
async_logger.addHandler(logging.handlers.QueueHandler(log_queue))
async_logger.setLevel(logging.INFO)

def heavy_inference_with_log():
    # 비동기 로깅은 이 함수(메인 로직)의 실행 시간을 늦추지 않습니다.
    async_logger.info("비동기 로그 기록 시작")
    # 모델 추론 시뮬레이션
    time.sleep(0.1)
    async_logger.info("비동기 로그 기록 완료")

heavy_inference_with_log()
print("메인 로직이 로깅 완료를 기다리지 않고 즉시 종료되었습니다.")

# 프로그램 종료 시 리스너 정지
# listener.stop()

INFO:AsyncAudit:비동기 로그 기록 시작
2026-06-16 04:05:28,795 [INFO] 비동기 로그 기록 시작
INFO:AsyncAudit:비동기 로그 기록 완료
2026-06-16 04:05:28,901 [INFO] 비동기 로그 기록 완료


메인 로직이 로깅 완료를 기다리지 않고 즉시 종료되었습니다.


### 🏁 최종 총평 (Final MD Review)

**1. 기술적 완성도:**
본 문서는 모델의 '정확도'에 매몰되기 쉬운 초급 단계에서 벗어나, **'가용성(Availability)'**과 **'보안(Security)'**이라는 운영의 핵심 가치를 매우 잘 전달하고 있습니다. 특히 API Key를 '자물쇠'에 비유한 시나리오는 실무적인 위험성을 직관적으로 이해하게 돕습니다.

**2. 실무 적용점:**
단순한 이론 나열이 아니라, `API Key`, `JWT`, `TLS`, `CORS` 등 실제 배포 시 즉각적으로 적용해야 할 체크리스트를 명확히 제시하고 있습니다. 미디어 처리 시 '검사' 단계를 강조한 점은 데이터 오염이나 공격으로부터 시스템을 보호하는 실무 지침으로 훌륭합니다.

**3. 학습자에게 주는 메시지:**
"열린 API는 공짜 GPU"라는 문구는 인프라 비용 관리가 곧 서비스의 생존과 직결됨을 시사합니다. AI 엔지니어가 모델러를 넘어 **MLOps/DevOps**적인 마인드셋을 갖추는 데 최적화된 가이드입니다.

**결론:**
본 문서는 AI 서비스의 **문지기(Gatekeeper)** 역할을 수행하기 위한 필수 지침서이며, 구현된 코드 예시들을 바탕으로 실제 프로덕션 환경에 바로 응용할 수 있는 높은 실용성을 갖추고 있습니다.

### 총평 (General Review)

**문서 요약 및 평가:**
이 문서는 AI 모델을 실제 서비스 환경으로 가져갈 때 필수적인 **'보안(Security)'**과 **'안정적인 입출력 처리'**를 다루고 있습니다. 단순히 모델을 실행하는 단계를 넘어, 기업용 서비스에서 발생할 수 있는 '비용 폭탄'이나 '모델 도용' 문제를 방지하기 위한 핵심 전략을 잘 가이드하고 있습니다.

**핵심 포인트:**
1.  **인증의 필수성**: 인증 없는 API는 공짜 GPU와 다름없다는 비유를 통해 API Key, JWT 등의 필요성을 강조함.
2.  **보안 계층**: TLS 암호화와 시크릿 관리의 중요성을 언급하여 데이터 유출 방지를 지향함.
3.  **실무 적합성**: 이미지 처리와 같은 미디어 처리를 단순 데이터 전송이 아닌 '출입 검사' 관점에서 접근한 점이 우수함.

**결론:**
본 가이드는 모델 성능 최적화만큼이나 중요한 **운영(Operations)** 측면의 보안 기초를 탄탄하게 다루고 있어, 주니어 엔지니어가 실무에 투입되기 전 반드시 숙지해야 할 내용을 포함하고 있습니다.

### 작업 요약

1.  **`pypdf` 라이브러리 설치**: PDF 텍스트 추출을 위해 설치 완료.
2.  **PDF 텍스트 추출**: `/content/모델배포기초6.pdf` 파일의 33페이지 전체 텍스트를 추출하여 변수에 저장.
3.  **추출 결과 확인**: 성공적인 추출 확인을 위해 초기 500자를 출력.
4.  **텍스트 파일 저장**: 전체 내용을 `extracted_pdf_content.txt` 파일로 저장 완료.